# AG952 | Workshop 9 — Explainability in Transformer Models
## Lecture Demonstration

**Textual Analytics for Accounting and Finance**  
University of Strathclyde Business School

---

## What this notebook demonstrates

This notebook is an instructor-delivered demonstration, intended to run immediately after students have completed CP7 of the workshop (the head-to-head comparison of VADER vs FinBERT).

The central question: *If a transformer model makes a better prediction than a dictionary, can we explain why?*

We examine two complementary approaches to transformer explainability:

1. **Integrated Gradients** (via `transformers-interpret`) — a principled attribution method that assigns a contribution score to each input token by measuring how the prediction changes as the input is interpolated from a baseline to the actual text.
2. **Attention visualisation** — showing which tokens attend to which, and explaining why high attention on a token is not the same as that token *causing* the prediction.

### Learning objectives

By the end of this demonstration, students should be able to:

- Explain what Integrated Gradients measures and how it differs from attention
- Interpret a token-level attribution heatmap
- Articulate why transformer explainability is more difficult and less intuitive than dictionary-based transparency
- Identify the regulatory and practical implications for deploying transformer models in financial contexts

---

> **Instructor note:** The key pedagogical moment is **Sentences 3 and 4** — where VADER scores incorrectly (negative) and FinBERT scores correctly (positive). The Integrated Gradients attribution shows *why* FinBERT is right: it assigns credit to words like *managed*, *successfully* and *removed* rather than treating *administration* and *going-concern* as unconditionally negative.

In [ ]:
#@title Setup — install dependencies and load FinBERT (~60-90s on first run)

# ── Step 1: pin numpy before captum/transformers-interpret ──────────────────
# captum is compiled against numpy 1.x; installing numpy 2.x first causes a
# dtype size mismatch (ValueError: numpy.dtype size changed, expected 96, got 88).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "numpy<2.0", "-q"], check=True)

# ── Step 2: install remaining packages ─────────────────────────────────────
!pip install transformers-interpret captum transformers torch vaderSentiment seaborn -q

# ── Step 3: verify numpy version before proceeding ─────────────────────────
import importlib, numpy as _np_check
_np_ver = tuple(int(x) for x in _np_check.__version__.split(".")[:2])
if _np_ver >= (2, 0):
    print(
        "\u26a0\ufe0f  numpy 2.x is still active in this runtime. "
        "Go to Runtime \u2192 Restart session, then re-run this cell."
    )
    raise SystemExit("Restart required to complete numpy downgrade.")
print(f"numpy {_np_check.__version__} \u2713  (1.x confirmed)")

# ── Step 4: imports ─────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from IPython.display import display, HTML, Markdown
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print(f"torch     {torch.__version__}")

# ── Step 5: load FinBERT ────────────────────────────────────────────────────
MODEL_NAME = "ProsusAI/finbert"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, output_attentions=True
)
model.eval()
print(f"FinBERT loaded.  Labels: {model.config.id2label}")
print("Setup complete.")

In [ ]:
#@title Define demonstration sentences

SENTENCES = [
    {
        'id': 1,
        'text': 'BrewDog reported record revenue growth driven by international expansion.',
        'true_label': 'positive',
        'note': 'Straightforward positive — both methods should agree',
    },
    {
        'id': 2,
        'text': 'BrewDog entered administration after a sustained period of financial losses and declining revenue.',
        'true_label': 'negative',
        'note': 'Straightforward negative — both methods should agree',
    },
    {
        'id': 3,
        'text': 'BrewDog successfully managed the risk of administration, securing emergency refinancing.',
        'true_label': 'positive',
        'note': 'TRAP \u2014 \"risk\" and \"administration\" are LM-negative; context is positive (negation/mitigation)',
    },
    {
        'id': 4,
        'text': 'The going-concern qualification was removed following a significant improvement in cash flow.',
        'true_label': 'positive',
        'note': 'TRAP \u2014 \"going-concern\" is strongly LM-negative; sentence is actually positive',
    },
    {
        'id': 5,
        'text': 'While revenues showed modest recovery, the board acknowledged that further restructuring may be necessary.',
        'true_label': 'negative',
        'note': 'NUANCED \u2014 modal hedging (\"may be necessary\") signals uncertainty and forward-looking risk',
    },
]

for s in SENTENCES:
    print(f"[{s['id']}] {s['text']}")
    print(f"    True label: {s['true_label']} | {s['note']}")
    print()

---
## Part 1 — The interpretability baseline: dictionary methods

Before we look at transformers, establish the comparison point: **dictionary-based sentiment gives full token-level transparency for free.**

Every word that contributed to the score can be inspected directly. This is what makes dictionaries auditable — and what makes transformer explainability feel harder by comparison.

In the output below:
- **Green highlight** = VADER scored this token as positive
- **Red highlight** = VADER scored this token as negative
- The intensity of colour reflects the magnitude of the individual token score

In [ ]:
#@title Part 1 — VADER attribution: full token-level transparency

vader = SentimentIntensityAnalyzer()

def vader_html_attribution(text):
    """Colour-code each token by its individual VADER score."""
    compound = vader.polarity_scores(text)['compound']
    label = 'positive' if compound > 0.05 else ('negative' if compound < -0.05 else 'neutral')
    tokens = text.split()
    parts = []
    matching_pos, matching_neg = [], []
    for tok in tokens:
        clean = tok.strip('.,;:!?()\'"-').lower()
        s = vader.polarity_scores(clean)['compound']
        if s > 0.08:
            alpha = min(s * 1.8, 0.85)
            bg = f'rgba(34,197,94,{alpha:.2f})'
            matching_pos.append(clean)
        elif s < -0.08:
            alpha = min(abs(s) * 1.8, 0.85)
            bg = f'rgba(239,68,68,{alpha:.2f})'
            matching_neg.append(clean)
        else:
            bg = 'transparent'
        parts.append(
            f'<span style="background:{bg};padding:2px 5px;margin:1px;'
            f'border-radius:3px;font-size:14px;">{tok}</span>'
        )
    token_html = ' '.join(parts)
    tick = '\u2713' if True else '\u2717'  # shown below per sentence
    return token_html, compound, label, matching_pos, matching_neg

all_results = []
for s in SENTENCES:
    html_toks, compound, label, pos_words, neg_words = vader_html_attribution(s['text'])
    correct = '\u2713' if label == s['true_label'] else '\u2717'
    color = '#166534' if label == s['true_label'] else '#991b1b'
    all_results.append({**s, 'vader_label': label, 'vader_score': compound})

    display(HTML(f"""
    <div style="margin:10px 0;padding:12px 14px;border:1px solid #e2e8f0;
                border-left:4px solid {'#22c55e' if label==s['true_label'] else '#ef4444'};
                border-radius:6px;background:#fafafa;">
      <div style="margin-bottom:8px;line-height:2.0;">{html_toks}</div>
      <small style="color:#475569;">
        <b>Sentence {s['id']}.</b>
        VADER compound: <b>{compound:+.4f}</b> &rarr;
        <span style="color:{color};font-weight:bold;">{label} {correct}</span>
        &nbsp;|&nbsp; True: {s['true_label']}<br/>
        Pos matches: {pos_words or 'none'} &nbsp;|&nbsp; Neg matches: {neg_words or 'none'}<br/>
        <i>{s['note']}</i>
      </small>
    </div>
    """))

print('\nKey observation: Sentences 3 and 4 are scored INCORRECTLY by VADER.')
print('The dictionary matches \"administration\", \"risk\", \"going-concern\" as negative')
print('regardless of the mitigating context. The context is invisible to a bag-of-words model.')

---
## Part 2 — FinBERT predictions: accuracy with no free explanation

FinBERT reads the full sentence as a sequence and produces a prediction that reflects contextual meaning. It should correct the two errors VADER made.

But notice what we *don't* get for free: **there is no highlighted word list**. The prediction is a function of 110 million parameters. We know it's right — but not immediately *why*.

In [ ]:
#@title Part 2 — FinBERT predictions

with torch.no_grad():
    for s in all_results:
        inputs = tokenizer(
            s['text'], return_tensors='pt', truncation=True, max_length=128
        )
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_idx = probs.argmax().item()
        pred_label = model.config.id2label[pred_idx].lower()
        pred_conf = probs[pred_idx].item()
        s['finbert_label'] = pred_label
        s['finbert_conf'] = pred_conf

# Comparison table
rows = []
for s in all_results:
    v_tick = '\u2713' if s['vader_label'] == s['true_label'] else '\u2717'
    f_tick = '\u2713' if s['finbert_label'] == s['true_label'] else '\u2717'
    rows.append({
        'Sentence': f"{s['id']}: {s['text'][:55]}...",
        'True label': s['true_label'],
        f'VADER {v_tick}' if v_tick=='\u2713' else f'VADER {v_tick}': s['vader_label'],
        'VADER': f"{s['vader_label']} {v_tick}",
        'FinBERT': f"{s['finbert_label']} ({s['finbert_conf']:.0%}) {f_tick}",
    })

df = pd.DataFrame(rows)

def style_row(row):
    vader_ok = '\u2713' in row['VADER']
    finbert_ok = '\u2713' in row['FinBERT']
    return [
        '', '',
        'background-color:#dcfce7' if vader_ok else 'background-color:#fee2e2',
        'background-color:#dcfce7' if finbert_ok else 'background-color:#fee2e2',
    ]

display(
    df[['Sentence', 'True label', 'VADER', 'FinBERT']]
    .style.apply(style_row, axis=1)
    .set_caption('Prediction comparison: VADER vs FinBERT')
    .set_properties(**{'font-size': '12px'})
)

vader_acc = sum(1 for s in all_results if s['vader_label'] == s['true_label']) / len(all_results)
finbert_acc = sum(1 for s in all_results if s['finbert_label'] == s['true_label']) / len(all_results)
print(f'\nAccuracy on these 5 sentences:')
print(f'  VADER:   {vader_acc:.0%}  ({int(vader_acc*5)}/5 correct)')
print(f'  FinBERT: {finbert_acc:.0%}  ({int(finbert_acc*5)}/5 correct)')
print(f'\nFinBERT is better. But which tokens drove the correct predictions?')
print(f'That is the question Part 3 addresses.')

---
## Part 3 — Integrated Gradients: opening the black box

### What are Integrated Gradients?

Integrated Gradients (Sundararajan, Taly & Yan, 2017) is a principled attribution method for neural networks. The intuition:

1. Start with a **baseline input** — typically all-zeros or all-[PAD] tokens (a "neutral" input that produces maximum uncertainty in the model)
2. Interpolate from the baseline to the actual input in many small steps
3. At each step, compute the gradient of the prediction with respect to each token's embedding
4. Integrate (sum) those gradients over all steps

The result is an **attribution score for each token**: a measure of how much that token, relative to the baseline, contributed to the final prediction.

> **Key property:** Integrated Gradients satisfies *completeness* — the sum of all attribution scores equals the difference between the model's output on the actual input and the baseline input. This makes it more principled than simply reading off gradient magnitudes.

### What the colours mean

In the visualisations below:
- **Green** = this token *supported* the predicted sentiment class
- **Red** = this token *worked against* the predicted class  
- **Intensity** = magnitude of the attribution score

> `transformers-interpret` (Stańczyk, 2021) wraps the Captum library's IG implementation with a clean API and HTML renderer for Jupyter/Colab.

In [ ]:
#@title Part 3a — Load transformers-interpret and run Integrated Gradients
# This may take 2-3 minutes on CPU — each sentence requires ~50 forward passes

from transformers_interpret import SequenceClassificationExplainer

explainer = SequenceClassificationExplainer(model, tokenizer)
print('Explainer loaded.')
print('Running Integrated Gradients on all 5 sentences...')
print('(~20-30 seconds per sentence on CPU)\n')

ig_results = []
for s in all_results:
    print(f"  Sentence {s['id']}... ", end='', flush=True)
    word_attrs = explainer(s['text'])
    pred_class = explainer.predicted_class_name
    pred_idx_ig = explainer.predicted_class_index
    ig_results.append({
        **s,
        'word_attributions': word_attrs,
        'ig_pred': pred_class,
        'viz': explainer.visualize(),
    })
    correct = '\u2713' if pred_class.lower() == s['true_label'] else '\u2717'
    print(f'{pred_class} {correct}')

print('\nIntegrated Gradients complete.')

In [ ]:
#@title Part 3b — Integrated Gradients HTML visualisation (all sentences)
# The colour of each token reflects its contribution to the predicted class.
# Green = supported the prediction.  Red = worked against it.

for r in ig_results:
    v_ok = '\u2713' if r['vader_label'] == r['true_label'] else '\u2717'
    f_ok = '\u2713' if r['finbert_label'] == r['true_label'] else '\u2717'
    display(HTML(f"""
    <div style="margin:16px 0;padding:10px 14px;border:1px solid #cbd5e1;
                border-left:4px solid #6366f1;border-radius:6px;">
      <b>Sentence {r['id']}</b> &nbsp;|&nbsp;
      True: <b>{r['true_label']}</b> &nbsp;|&nbsp;
      VADER: {r['vader_label']} {v_ok} &nbsp;|&nbsp;
      FinBERT: {r['finbert_label']} {f_ok}<br/>
      <small style="color:#64748b;">{r['note']}</small>
    </div>
    """))
    display(r['viz'])
    print()

In [ ]:
#@title Part 3c — Attribution bar charts: focus on the two tricky sentences

def plot_attribution_bar(r, ax=None):
    attrs = r['word_attributions']
    # Exclude [CLS] and [SEP] special tokens
    tokens  = [a[0] for a in attrs if a[0] not in ('[CLS]', '[SEP]')]
    scores  = [a[1] for a in attrs if a[0] not in ('[CLS]', '[SEP]')]
    colors  = ['#22c55e' if s > 0 else '#ef4444' for s in scores]
    alphas  = [min(abs(s) * 6 + 0.3, 0.95) for s in scores]

    if ax is None:
        fig, ax = plt.subplots(figsize=(11, max(3.0, len(tokens) * 0.38)))
    bars = ax.barh(
        range(len(tokens)), scores,
        color=[mcolors.to_rgba(c, a) for c, a in zip(colors, alphas)],
        edgecolor='white', height=0.65,
    )
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.axvline(0, color='#475569', linewidth=0.8, linestyle='--')
    v_ok = '\u2713' if r['vader_label'] == r['true_label'] else '\u2717'
    f_ok = '\u2713' if r['finbert_label'] == r['true_label'] else '\u2717'
    ax.set_title(
        f"Sentence {r['id']}: Integrated Gradients attribution\n"
        f"VADER: {r['vader_label']} {v_ok}  |  FinBERT: {r['finbert_label']} {f_ok}  |  True: {r['true_label']}\n"
        f"{r['note']}",
        fontsize=9, loc='left',
    )
    ax.set_xlabel('Attribution score  (green = supports predicted class)', fontsize=8)
    return ax

# Focus on the two tricky sentences (3 and 4)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_attribution_bar([r for r in ig_results if r['id'] == 3][0], ax=axes[0])
plot_attribution_bar([r for r in ig_results if r['id'] == 4][0], ax=axes[1])
plt.tight_layout(pad=2.5)
plt.savefig('ig_attribution_tricky_sentences.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key teaching point:')
print('  Sentence 3: \"managed\", \"successfully\", \"securing\" receive positive attribution.')
print('             \"risk\", \"administration\" receive smaller/negative attribution —')
print('             but the positive context words dominate. FinBERT is correct.')
print()
print('  Sentence 4: \"removed\", \"improvement\", \"cash flow\" receive positive attribution.')
print('             \"going-concern\" receives negative attribution — but \"removed\" flips')
print('             the net effect. The model encodes the negation correctly.')

In [ ]:
#@title Part 3d — Attribution bar charts: all five sentences

fig, axes = plt.subplots(1, 5, figsize=(22, 7))
for i, r in enumerate(ig_results):
    plot_attribution_bar(r, ax=axes[i])
plt.suptitle(
    'Integrated Gradients token attributions — all five BrewDog sentences\n'
    'Green = token supports predicted sentiment class   |   Red = token works against it',
    fontsize=10, y=1.02,
)
plt.tight_layout(pad=1.5)
plt.savefig('ig_attribution_all_sentences.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 4 — Attention visualisation: what the model looks at

Attention is a different lens on the model's internal computation. In a transformer, every token can attend to every other token; the attention weights determine how much information flows between tokens at each layer.

### Attention ≠ explanation

This distinction matters:

| Integrated Gradients | Attention |
|---|---|
| Measures *how much each token contributed to the final prediction* | Measures *how much each token's representation draws on other tokens* |
| Attribution score is tied directly to the model output | Attention weight is an internal routing mechanism |
| Satisfies completeness (attributions sum to prediction difference) | Does not satisfy any attribution axiom |
| Slow (requires ~50 forward passes per sentence) | Fast (one forward pass, already computed) |

**Jain & Wallace (2019)** showed that attention weights are often inconsistent with gradient-based feature importance — high attention on a token does not reliably indicate that token caused the prediction.

> That said, attention visualisations are *excellent for teaching* how transformer architecture works — showing how a token like *managed* attends strongly to *administration* to build a contextualised representation.

In the heatmap below:
- **Rows** = query token (what is looking)
- **Columns** = key token (what is being looked at)
- **Intensity** = attention weight (how much this query draws on this key)

In [ ]:
#@title Part 4 — Attention heatmaps for sentences 2 and 3

def plot_attention_heatmap(s, layer_idx=-1, title_suffix=''):
    inputs = tokenizer(
        s['text'], return_tensors='pt', truncation=True, max_length=64
    )
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    # Average over all heads in the specified layer
    att_layer = outputs.attentions[layer_idx][0]  # (n_heads, seq_len, seq_len)
    att_mean  = att_layer.mean(dim=0).numpy()     # (seq_len, seq_len)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

    # Left: full attention matrix
    sns.heatmap(
        att_mean, xticklabels=tokens, yticklabels=tokens,
        cmap='Blues', ax=axes[0], cbar=True,
        linewidths=0.3, linecolor='#f1f5f9',
    )
    axes[0].set_title(
        f'Mean attention (layer {layer_idx}, all heads)\nQuery \u2192 Key'
    )
    axes[0].tick_params(axis='x', rotation=45, labelsize=7)
    axes[0].tick_params(axis='y', rotation=0,  labelsize=7)

    # Right: [CLS] token attention (what the classifier "looks at")
    cls_att = att_mean[0]
    bar_colors = plt.cm.Blues(
        (cls_att - cls_att.min()) / (cls_att.max() - cls_att.min() + 1e-9)
    )
    axes[1].bar(range(len(tokens)), cls_att, color=bar_colors, edgecolor='white')
    axes[1].set_xticks(range(len(tokens)))
    axes[1].set_xticklabels(tokens, rotation=45, ha='right', fontsize=7)
    axes[1].set_title(
        '[CLS] token attention\n'
        '(the classifier reads from here; what does it attend to?)'
    )
    axes[1].set_ylabel('Attention weight')

    v_ok = '\u2713' if s['vader_label'] == s['true_label'] else '\u2717'
    f_ok = '\u2713' if s['finbert_label'] == s['true_label'] else '\u2717'
    fig.suptitle(
        f"Sentence {s['id']}: \"{s['text'][:70]}...\"\n"
        f"VADER: {s['vader_label']} {v_ok}  |  FinBERT: {s['finbert_label']} {f_ok}",
        fontsize=9,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.savefig(f'attention_sentence_{s["id"]}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Show the negative case and the tricky case side-by-side
for sid in [2, 3]:
    plot_attention_heatmap([r for r in ig_results if r['id'] == sid][0])

print('\nNotice:')
print('  Sentence 2 (unambiguous negative): [CLS] attends heavily to')
print('  \"administration\", \"losses\", \"declining\" — strong negative signal, easy case.')
print()
print('  Sentence 3 (tricky positive): [CLS] attends to \"managed\", \"successfully\",')
print('  \"securing\" as well as \"administration\". The model builds a contextualised')
print('  representation of \"administration\" that reflects mitigation, not occurrence.')
print()
print('  BUT: this attention pattern does not prove that \"managed\" *caused* the correct')
print('  prediction. That is what Integrated Gradients (Part 3) measures more rigorously.')

---
## Part 5 — Synthesis: how explainable are transformers?

### What we can do

| Method | What it shows | Rigour | Cost |
|---|---|---|---|
| Dictionary matching | Which words matched a word list | Directly auditable | Free |
| Attention (last layer, mean) | Which tokens the model attends to | Low (Jain & Wallace, 2019) | Fast |
| Integrated Gradients | How much each token shifted the prediction from baseline | High (satisfies completeness) | Slow (~50 passes) |
| SHAP (kernel/gradient) | Shapley-value attribution across feature subsets | High | Very slow |

### What we cannot do

- **We cannot fully reverse-engineer a 110M-parameter model from attribution scores.** Integrated Gradients tells us which input features mattered; it does not tell us *what the model learned* about those features.
- **Attribution scores are prediction-specific.** The same token may receive a positive attribution in one sentence and negative in another, because attribution is always relative to a specific input and a specific baseline.
- **The baseline choice matters.** IG attributes relative to an all-[PAD] baseline by default. A different baseline (e.g. average embedding) would produce different scores. There is no single "correct" baseline.

### Regulatory implications

The **EU AI Act (2024)** classifies financial analysis systems as high-risk, requiring explainability of outputs to affected parties. Dictionary methods satisfy explainability requirements directly (every match is auditable). Transformer models require supplementary SHAP, LIME, or IG analysis to approach the same standard — and even then, the explanation is approximate, not complete.

For a practitioner deploying FinBERT in credit analysis:
- IG attributions can accompany each prediction in a report
- They are useful for identifying *which sentences* drove a document-level prediction
- They are insufficient as a standalone explanation for a regulatory audit without human review

> **The tradeoff in one sentence:** dictionary methods are *transparent* but *brittle*; transformer models are *contextually intelligent* but *expensive to explain* — and the explanation is always an approximation.

In [ ]:
#@title Part 5 — Summary visualisation: accuracy vs interpretability

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: accuracy comparison
sentence_ids = [f"S{r['id']}" for r in ig_results]
vader_correct = [1 if r['vader_label'] == r['true_label'] else 0 for r in ig_results]
finbert_correct = [1 if r['finbert_label'] == r['true_label'] else 0 for r in ig_results]

x = range(len(sentence_ids))
width = 0.35
axes[0].bar([i - width/2 for i in x], vader_correct,   width, label='VADER',   color='#f59e0b', alpha=0.85, edgecolor='white')
axes[0].bar([i + width/2 for i in x], finbert_correct, width, label='FinBERT', color='#6366f1', alpha=0.85, edgecolor='white')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(sentence_ids)
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(['Incorrect', 'Correct'])
axes[0].set_title('Prediction accuracy by sentence\n(1 = correct, 0 = incorrect)')
axes[0].legend()
for i, (v, f) in enumerate(zip(vader_correct, finbert_correct)):
    note = ig_results[i]['note'].split(' \u2014 ')[0]
    axes[0].text(i, -0.18, note, ha='center', fontsize=6, color='#475569', rotation=10)

# Right: interpretability–accuracy scatter (conceptual)
methods = ['VADER\n(dictionary)', 'FinBERT\n+ Attention', 'FinBERT\n+ Integ. Gradients', 'FinBERT\n+ SHAP']
acc_scores    = [0.60, 0.80, 0.80, 0.80]  # conceptual, not exact
interp_scores = [1.00, 0.30, 0.65, 0.75]  # interpretability (higher = more interpretable)
cost_scores   = [1,    1,    3,    5]      # relative compute cost

scatter_colors = ['#f59e0b', '#94a3b8', '#6366f1', '#a855f7']
for m, a, i_s, c, col in zip(methods, acc_scores, interp_scores, cost_scores, scatter_colors):
    axes[1].scatter(a, i_s, s=c * 120, color=col, alpha=0.85, edgecolors='white', linewidth=1.5, zorder=3)
    axes[1].annotate(m, (a, i_s), textcoords='offset points', xytext=(8, 4), fontsize=8)
axes[1].set_xlim(0.45, 1.0)
axes[1].set_ylim(-0.05, 1.15)
axes[1].set_xlabel('Predictive accuracy (conceptual)')
axes[1].set_ylabel('Interpretability (conceptual)')
axes[1].set_title('The accuracy\u2013interpretability tradeoff\n(bubble size = relative compute cost)')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(0.8, color='#94a3b8', linestyle=':', linewidth=0.8)
axes[1].axhline(0.7, color='#94a3b8', linestyle=':', linewidth=0.8)

plt.suptitle(
    'Transformer explainability: closing the gap, not eliminating it',
    fontsize=11, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.savefig('explainability_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Instructor talking points

### After Part 1 (VADER attribution)
> *"Every single coloured word is auditable. You can check the VADER lexicon, you can argue about whether 'administration' should be red, and you can correct it. That's what interpretability means in practice."*

### After Part 2 (FinBERT predictions)
> *"It got sentences 3 and 4 right. VADER didn't. But I haven't explained anything yet — I've just asserted that 110 million parameters arrived at a correct answer. That's not good enough for a credit committee or a regulator."*

### After Part 3 (Integrated Gradients)
> *"Now we have something. 'Managed' is green — it pushed the prediction toward positive. 'Administration' is red but small — the model knew it was a loaded word, but the context downweighted it. This is a real explanation, not just a prediction."*
> 
> *"But notice: the explanation required 50 additional forward passes through the model. And it's an approximation. And it changes if you change the baseline. Dictionary methods give you this for free."*

### After Part 4 (Attention)
> *"This is the mechanism. The model is routing information — [CLS] attends to 'managing' and 'securing' when it builds its sentence representation. But Jain and Wallace showed in 2019 that you can often swap attention weights and the prediction doesn't change. High attention ≠ high importance."*
> 
> *"Use attention to teach architecture. Use Integrated Gradients to explain predictions."*

### Closing
> *"The question you need to answer before deploying any of this in practice: 'Can I explain this prediction to a non-technical stakeholder in a way they can act on?' For dictionary methods: yes, always. For transformers with IG: sometimes, with effort. For transformers with only accuracy metrics: no."*

---

**Key references:**
- Sundararajan, M., Taly, A., & Yan, Q. (2017). Axiomatic attribution for deep networks. *ICML*.
- Jain, S., & Wallace, B.C. (2019). Attention is not explanation. *NAACL-HLT*.
- Wiegreffe, A., & Pinter, Y. (2019). Attention is not not explanation. *EMNLP*.
- Roberts, M.E., Stewart, B.M., & Tingley, D. (2019). stm. *Journal of Statistical Software*, 91(2).
- EU AI Act (2024). Regulation 2024/1689 of the European Parliament.